In [1]:
import numpy as np
import sisl
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from sisl import viz

# Automation using `kind = 'armchair'`

In [ ]:
BOND = 1.42  # C-C bond length in Angstrom
KIND = 'armchair'  # 'armchair' or 'zigzag'
VACUUM = 3.0  # Vacuum layer in Angstrom
WIDTH = 4 # Width of the nanoribbon in number of dimer lines
LENGTH = 5 # Length of the nanoribbon in number of unit cells

def find_center_hexagon(structure, manual_center=None):
    """Find hexagon of atoms closest to the geometric center of the structure.  
    Returns the indices of the 6 atoms forming the hexagon.
    """
    coordinates = structure.xyz # Atomic coordinates
    if manual_center is None:
        geom_center = structure.center() # Geometric center of the structure
    else:
        geom_center = np.array(manual_center)
    diff = geom_center - coordinates # Difference vectors from geometric center to each atom
    distances_from_center = np.linalg.norm(diff, axis=1) # Euclidean distances from geometric center
    atoms_idx_sorted_by_distance = np.argsort(distances_from_center) # Indices of atoms sorted by distance from geometric center
    return atoms_idx_sorted_by_distance[:6], geom_center # Return indices of the 6 closest atoms forming a hexagon

def find_interatomic_distances_in_hexagon(structure):
    """Find inter-atomic distances in the hexagon closest to the geometric center."""
    center_hexagon_idx, _ = find_center_hexagon(structure)
    hexagon_coords = structure.xyz[center_hexagon_idx]
    interatomic_distances = np.linalg.norm(hexagon_coords[:, None, :] - hexagon_coords[None, :, :], axis=-1)
    return interatomic_distances

def is_hexagon(structure):
    """Check if the structure contains hexagons."""
    proposed_center_idx, _ = find_center_hexagon(structure)
    flake = sisl.geom.graphene_flake(0, bond=1.42)
    inter_dist_in_hexagon = find_interatomic_distances_in_hexagon(flake)
    max_distance_in_hexagon = np.max(inter_dist_in_hexagon)
    
    proposed_center_hexagon = structure.xyz[proposed_center_idx]
    if not np.allclose(proposed_center_hexagon, inter_dist_in_hexagon, atol=0.1):
        return False
    else:
        return True



def generate_structure(width=2, length=5, **KWARGS):
    
    bond = KWARGS.get('bond', 1.42)
    kind = KWARGS.get('kind', 'armchair')
    vacuum = KWARGS.get('vacuum', 3.0)
    base = sisl.geom.graphene_nanoribbon(width=width, bond=bond, kind=kind, vacuum=vacuum)
    base = base.repeat(length, axis=0)
    structure = base.copy()
    center_hexagon_idx, geom_center = find_center_hexagon(structure)
    center_hex = structure.xyz[center_hexagon_idx]
    origin_for_rotation = center_hex.mean(axis=0) # find center of hexagon closest to geometric center
    for i in range(1, 3):
        angle = i * 60
        rotated = base.rotate(angle=angle, v=[0,0,1], origin=origin_for_rotation)
        structure += rotated
    return structure

def plot_with_center(structure):
    center_hexagon_idx, geom_center = find_center_hexagon(structure)
    
    geom_center_scatter = go.Scatter(x=[geom_center[0]], 
                                     y=[geom_center[1]], 
                                     mode='markers', 
                                     marker=dict(size=8, color='red'), 
                                     name='Geometric Center')
    
    fig = structure.plot(axes="xy", bind_bonds_to_ats=True)  # Plot the structure in the xy-plane
    center_hex_atoms = {
        "color": "red",
        "atoms": center_hexagon_idx.tolist(),
        # "name": "Center Hexagon"
    }
    fig.update_inputs(atoms_style = center_hex_atoms)
    fig.add_trace(geom_center_scatter)
    return fig

graphene = generate_structure(width=WIDTH, length=LENGTH, bond=BOND, kind=KIND, vacuum=VACUUM)
# graphene.plot(axes="yx")
fig = plot_with_center(graphene)
fig.update_inputs(axes="yx")
fig.show()

In [ ]:


flake = sisl.geom.graphene_flake(0)

flake.plot(axes="xy")
inter_dist = find_interatomic_distances_in_hexagon(flake)
print("Inter-atomic distances in the identified hexagon (should be close to 1.42 Angstrom):")
print(inter_dist)

Inter-atomic distances in the identified hexagon (should be close to 1.42 Angstrom):
[[0.         1.42       2.45951215 1.42       2.45951215 2.84      ]
 [1.42       0.         1.42       2.45951215 2.84       2.45951215]
 [2.45951215 1.42       0.         2.84       2.45951215 1.42      ]
 [1.42       2.45951215 2.84       0.         1.42       2.45951215]
 [2.45951215 2.84       2.45951215 1.42       0.         1.42      ]
 [2.84       2.45951215 1.42       2.45951215 1.42       0.        ]]
